# Task 1I - Exploratory Data Analysis (EDA)

### Objective
To perform exploratory data analysis on the Olist e-commerce dataset and find useful patterns, trends and relationships in the data.

**Libraries used:** Pandas, NumPy, Matplotlib

**Dataset:** Olist Brazilian E-Commerce Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# loading the datasets
orders = pd.read_csv("olist_data/olist_orders_dataset.csv")
payments = pd.read_csv("olist_data/olist_order_payments_dataset.csv")
items = pd.read_csv("olist_data/olist_order_items_dataset.csv")
reviews = pd.read_csv("olist_data/olist_order_reviews_dataset.csv")

print("Orders:", orders.shape)
print("Payments:", payments.shape)
print("Items:", items.shape)
print("Reviews:", reviews.shape)

## 1. Checking the data

In [ ]:
orders.head()

In [ ]:
print("Missing values in Orders:")
print(orders.isnull().sum())

print("\nDuplicate rows:", orders.duplicated().sum())

## 2. Preparing the data

In [ ]:
orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"], errors="coerce"
)
orders["purchase_month"] = orders["order_purchase_timestamp"].dt.to_period("M").astype(str)

# total payment for each order
payment_agg = payments.groupby("order_id").agg(
    payment_value=("payment_value", "sum"),
    payment_installments=("payment_installments", "max")
).reset_index()

# item information for each order
item_agg = items.groupby("order_id").agg(
    item_count=("order_item_id", "count"),
    freight_value=("freight_value", "sum"),
    item_value=("price", "sum")
).reset_index()

# one review score for each order
review_agg = reviews[["order_id", "review_score"]].drop_duplicates("order_id")

# merging the useful columns
eda = orders.merge(payment_agg, on="order_id", how="left")
eda = eda.merge(item_agg, on="order_id", how="left")
eda = eda.merge(review_agg, on="order_id", how="left")

print("Final dataset shape:", eda.shape)

## 3. Descriptive statistics

In [ ]:
eda[["payment_value", "item_value", "freight_value",
       "item_count", "payment_installments", "review_score"]].describe()

## 4. Order status

In [ ]:
status_count = eda["order_status"].value_counts()
print(status_count)

status_count.plot(kind="bar", figsize=(9,5))
plt.title("Orders by Status")
plt.xlabel("Order Status")
plt.ylabel("Number of Orders")
plt.xticks(rotation=45)
plt.show()

## 5. Monthly order trend

In [ ]:
monthly_orders = eda.groupby("purchase_month").size()

plt.figure(figsize=(11,5))
monthly_orders.plot(marker="o")
plt.title("Monthly Order Trend")
plt.xlabel("Month")
plt.ylabel("Number of Orders")
plt.xticks(rotation=60)
plt.show()

## 6. Payment value distribution

In [ ]:
# The top 1% of values are removed only for this plot
# so that the main distribution is easier to see.
payment_plot = eda["payment_value"].dropna()
payment_plot = payment_plot[payment_plot <= payment_plot.quantile(0.99)]

plt.figure(figsize=(9,5))
plt.hist(payment_plot, bins=30)
plt.title("Distribution of Order Payment Value")
plt.xlabel("Payment Value")
plt.ylabel("Frequency")
plt.show()

## 7. Review score analysis

In [ ]:
review_count = eda["review_score"].value_counts().sort_index()
print(review_count)

review_count.plot(kind="bar", figsize=(8,5))
plt.title("Customer Review Scores")
plt.xlabel("Review Score")
plt.ylabel("Number of Reviews")
plt.xticks(rotation=0)
plt.show()

## 8. Relationship between payment and review score

In [ ]:
temp = eda[["payment_value", "review_score"]].dropna()

if len(temp) > 5000:
    temp = temp.sample(5000, random_state=42)

plt.figure(figsize=(9,5))
plt.scatter(temp["payment_value"], temp["review_score"], alpha=0.3)
plt.title("Payment Value vs Review Score")
plt.xlabel("Payment Value")
plt.ylabel("Review Score")
plt.show()

print("Correlation:",
      eda[["payment_value", "review_score"]].corr().iloc[0,1])

## 9. Key observations

From the analysis, I observed the following:

1. The Olist order dataset contains **99,441 orders**.
2. **delivered** is the most common order status, with **96,478 orders**.
3. The average order payment value is about **160.99**, while the median is **105.29**.
4. The average customer review score is **4.09 out of 5**.
5. The month with the highest number of orders in this dataset is **2017-11**.
6. The correlation between payment value and review score is approximately **-0.049**, so there is not a strong linear relationship between these two variables.
7. The charts help to understand the order trends, payment distribution and customer review patterns.

## 10. Conclusion

EDA helped me understand the basic structure of the Olist e-commerce data before applying any machine learning techniques. I analyzed missing values, descriptive statistics, order status, monthly order trends, payment values and review scores.

The analysis shows useful patterns in customer orders and payments and also demonstrates how visualizations can be used to identify trends and relationships in real-world data.